# SFT Training V126: Unsloth + Boxed Loss Weight 5x (Based on V120)

**Strategy:**
- **Single-Stage**: 5200 balanced data (all 7 types) with thinking CoT
- **BOXED_LOSS_WEIGHT = 5.0** — upweight tokens after last `\boxed{` by 5x in loss
- **Unsloth Framework**: Uses `FastLanguageModel` for Mamba gate_proj/x_proj LoRA
- Includes Unsloth→HF PEFT adapter conversion pipeline for submission
- Stage 2 disabled

**Key differences from V120 (HF PEFT, 0.72):**
- Unsloth splits Mamba `in_proj` into `gate_proj` + `x_proj` → separate LoRA → more Mamba parameters
- Unsloth fuses MoE expert weights (w1/w2) → efficient 3D LoRA
- All other hyperparameters identical to V120
- Same boxed loss weight 5x, same data, same format

**Critical**: User prompt suffix matches official evaluation **EXACTLY**.

In [ ]:
import subprocess, sys, os, glob

# ================================================================
# Strategy: All offline packages from hastws/sft-offline-packages
#   - Contains: unsloth, unsloth_zoo, trl, tyro, hf_transfer,
#     nest_asyncio, annotated_doc
#   - Other deps (transformers, peft, datasets, accelerate, torch,
#     bitsandbytes, triton, sentencepiece) already in Kaggle env
#   - No dependency on third-party datasets
# ================================================================

OFFLINE_DIRS = [
    "/kaggle/input/sft-offline-packages",
    "/kaggle/input/datasets/hastws/sft-offline-packages",
]

offline_dir = None
for d in OFFLINE_DIRS:
    if os.path.isdir(d):
        whl_files = [f for f in os.listdir(d) if f.endswith('.whl')]
        if whl_files:
            offline_dir = d
            print(f"Found offline packages at: {d} ({len(whl_files)} wheels)")
            break

if offline_dir:
    whls = sorted(glob.glob(os.path.join(offline_dir, "*.whl")))
    print(f"Installing {len(whls)} wheels with --no-deps...")
    for whl in whls:
        name = os.path.basename(whl)
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-deps", whl],
            capture_output=True, text=True, timeout=120
        )
        status = "OK" if r.returncode == 0 else "FAIL"
        print(f"  {status}: {name}")
        if r.returncode != 0 and "unsloth" in name.lower():
            print(f"    ERROR: {r.stderr[-200:]}")
else:
    print("WARNING: Offline packages not found, trying online install...")
    for pkg in ["unsloth", "unsloth_zoo", "trl"]:
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-deps", pkg],
            capture_output=True, text=True, timeout=300
        )
        print(f"  {'OK' if r.returncode == 0 else 'FAIL'}: {pkg}")

# --- flash_attn (from Kaggle env or utility script) ---
try:
    import flash_attn
    print(f"flash_attn={flash_attn.__version__}")
except ImportError:
    fa_whls = glob.glob("/kaggle/input/**/*flash_attn*.whl", recursive=True)
    if fa_whls:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", fa_whls[0]],
                       capture_output=True, text=True, timeout=120)
        print(f"Installed flash_attn from {os.path.basename(fa_whls[0])}")

# --- Verify key packages ---
print("\n=== Package Versions ===")
for pkg in ["unsloth", "unsloth_zoo", "trl", "peft", "datasets", "transformers"]:
    try:
        mod = __import__(pkg)
        print(f"  {pkg}={getattr(mod, '__version__', '?')}")
    except ImportError as e:
        if pkg == "unsloth":
            raise RuntimeError(f"FATAL: unsloth not importable: {e}")
        print(f"  WARNING: {pkg}: {e}")

print("\nPackage setup complete")


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys
import stat
import shutil
import gc
import zipfile
import time
import json
import math
import re
import polars as pl
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset

# NOTE: trl imports moved to training cell (after Unsloth import)
# to ensure Unsloth's monkey-patching takes effect

print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"GPU memory: {mem / 1024**3:.1f} GB")

# --- Triton / Blackwell environment fixes ---
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# Copy PTXAS binaries to writable temp — search multiple paths
_ptxas_candidates = [
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell",
    "/kaggle/input/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell",
]
# Also search recursively
import glob as _g
_ptxas_candidates += _g.glob("/kaggle/usr/**/ptxas*blackwell", recursive=True)
_ptxas_candidates += _g.glob("/kaggle/input/**/ptxas*blackwell", recursive=True)

src = None
for _candidate in _ptxas_candidates:
    if os.path.exists(_candidate):
        src = _candidate
        break

dst = "/tmp/ptxas-blackwell"
if src:
    print(f"Found ptxas-blackwell at: {src}")
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    print("Triton patched for Blackwell")
else:
    print("ptxas-blackwell not found, skipping Triton patch")

print("Imports and environment ready")

## Configuration

All hyperparameters are defined here — identical to V120 except framework.
The `PROMPT_SUFFIX` is copied verbatim from the **official evaluation metric script** — do not modify it.

In [ ]:
# =============================================
#  🔧 HYPERPARAMETERS — EDIT HERE
# =============================================

# --- Stage 1: Thinking + boxed (single-stage) ---
STAGE1_N_PER_TYPE = None   # Skip re-sampling (already balanced at 4400 by gen_thinking.py)
STAGE1_THINKING_ONLY = False  # False = include all rows
STAGE1_ANSWER_ONLY = False   # False = use thinking format with \boxed{}
STAGE1_DROP_LONG = True    # True = drop samples exceeding STAGE1_MAX_SEQ (zero truncation)
STAGE1_LR        = 1e-4
STAGE1_EPOCHS    = 2
STAGE1_MAX_SEQ   = 1792    # Tight margin over data max=1678 (~15% speedup, still zero truncation)
STAGE1_BATCH     = 1       # Per-device batch size
STAGE1_GRAD_ACCUM = 4      # Effective batch = 1 × 4 = 4
STAGE1_PACKING   = False    # V101: ultra-short CoT (~100 tok) packs extremely well
# --- Stage 2: Format polish (disabled for answer-only) ---
STAGE2_ENABLED   = False
STAGE2_N_SAMPLES = 600
STAGE2_LR        = 4e-5    # 1/5 of Stage 1
STAGE2_EPOCHS    = 1
STAGE2_MAX_SEQ   = 1024    # match Stage 1
STAGE2_BATCH     = 1
STAGE2_GRAD_ACCUM = 4
STAGE2_PACKING   = True

# --- LoRA (same as V120, dropout=0 to align with 0.81/0.85 reference) ---
LORA_RANK        = 32
LORA_ALPHA       = 64
LORA_DROPOUT     = 0.0

# --- Boxed Loss Weight ---
BOXED_LOSS_WEIGHT = 5.0   # Upweight tokens after last \boxed{ by Nx (1.0 = disabled)

# --- Type Filter ([] = all types, ['bit_ops'] = only bit_ops) ---
TRAIN_TYPES      = []  # All types (5200 rows: V3 verify CoT)

# --- Holdout Evaluation ---
HOLDOUT_ENABLED  = False
HOLDOUT_N_PER_TYPE = 10   # 10 per-type holdout samples

# --- Hardcoded holdout: empty = use random stratified sampling ---
HOLDOUT_HARD_IDS = []

# =============================================
#  🔴 OFFICIAL PROMPT SUFFIX — DO NOT MODIFY
# =============================================
# Source: competition_notebooks/nemotron-baseline-evaluation.ipynb
# Lines: user_content = item.prompt + '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)
EVAL_MAX_TOKENS  = 3584     # Match official max_tokens

print(f"Stage 1: n_per_type={STAGE1_N_PER_TYPE}, answer_only={STAGE1_ANSWER_ONLY}, drop_long={STAGE1_DROP_LONG}")
print(f"Stage 1: LR={STAGE1_LR}, epochs={STAGE1_EPOCHS}, max_seq={STAGE1_MAX_SEQ}")
print(f"Stage 1: batch={STAGE1_BATCH}, grad_accum={STAGE1_GRAD_ACCUM}, packing={STAGE1_PACKING}")
print(f"Stage 2: enabled={STAGE2_ENABLED}, n={STAGE2_N_SAMPLES}, LR={STAGE2_LR}, packing={STAGE2_PACKING}")
print(f"LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"Boxed loss weight: {BOXED_LOSS_WEIGHT}")
print(f"Train types: {TRAIN_TYPES if TRAIN_TYPES else 'ALL'}")
print(f"Holdout: enabled={HOLDOUT_ENABLED}, n_per_type={HOLDOUT_N_PER_TYPE}")
print(f"Eval max tokens: {EVAL_MAX_TOKENS}")
print(f"Prompt suffix: {repr(PROMPT_SUFFIX)}")

## Data Loading & Statistics

In [ ]:
print("=== DATA LOADING: sft_thinking.csv (9000 balanced) ===")  # Version marker

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
COMP_DATA = '/kaggle/input/nvidia-nemotron-3-reasoning-challenge'

# --- Exhaustive search for training data ---
import glob as _glob

TARGET_FILE = 'sft_thinking.csv'

# Method 1: Check known candidates
_COT_CANDIDATES = [
    '/kaggle/input/prog-cot-training-data',
    '/kaggle/input/datasets/hastws/prog-cot-training-data',
]
COT_DATA = None
for d in _COT_CANDIDATES:
    csv_path = os.path.join(d, TARGET_FILE)
    if os.path.isfile(csv_path):
        COT_DATA = d
        print(f"Found via candidate: {csv_path}")
        break

# Method 2: Glob search under /kaggle/input/
if COT_DATA is None:
    print("Candidate dirs failed. Searching /kaggle/input/ recursively...")
    matches = _glob.glob(f'/kaggle/input/**/{TARGET_FILE}', recursive=True)
    if matches:
        COT_DATA = os.path.dirname(matches[0])
        print(f"Found via glob: {matches[0]}")
    else:
        print(f"ERROR: {TARGET_FILE} not found anywhere under /kaggle/input/")
        print("\n--- /kaggle/input/ contents ---")
        for root, dirs, files in os.walk('/kaggle/input/'):
            level = root.replace('/kaggle/input/', '').count(os.sep)
            indent = ' ' * 2 * level
            print(f'{indent}{os.path.basename(root)}/')
            if level < 3:
                subindent = ' ' * 2 * (level + 1)
                for f in files[:20]:
                    print(f'{subindent}{f}')
        raise FileNotFoundError(f"{TARGET_FILE} not found under /kaggle/input/. See listing above.")

print(f"COT_DATA = {COT_DATA}")
print(f"  Files: {os.listdir(COT_DATA)[:10]}")

# Load the merged dataset
train_df = pl.read_csv(f'{COT_DATA}/{TARGET_FILE}')

print(f"{'='*60}")
print(f"  Loaded: {TARGET_FILE} — {len(train_df)} rows")
print(f"{'='*60}")

# Statistics
pdf = train_df.to_pandas()
has_thinking = pdf['thinking'].fillna('').str.strip().str.len() > 0
n_with = has_thinking.sum()
n_without = (~has_thinking).sum()
print(f"\n  With thinking: {n_with} ({n_with/len(pdf)*100:.1f}%)")
print(f"  Without thinking (answer-only): {n_without} ({n_without/len(pdf)*100:.1f}%)")

# Classify thinking types
short_mask = has_thinking & (pdf['thinking'].str.len() < 50)
long_mask = has_thinking & (pdf['thinking'].str.len() >= 50)
print(f"  - Compact rules (<50 chars): {short_mask.sum()}")
print(f"  - Full CoT (≥50 chars): {long_mask.sum()}")

# Show thinking length distribution
print(f"\n  Thinking length stats (non-empty):")
lengths = pdf.loc[has_thinking, 'thinking'].str.len()
print(f"    min={lengths.min()}, median={lengths.median():.0f}, mean={lengths.mean():.0f}, max={lengths.max()}")

# Check for any data issues
print(f"\n  --- Sanity checks ---")
print(f"  Empty prompt: {(pdf['prompt'].fillna('').str.len() == 0).sum()}")
print(f"  Empty answer: {(pdf['answer'].fillna('').astype(str).str.len() == 0).sum()}")
boxed_in_thinking = pdf.loc[has_thinking, 'thinking'].str.contains(r'\\boxed', regex=True, na=False).sum()
print(f"  \\boxed in thinking: {boxed_in_thinking}")
print(f"  Columns: {list(pdf.columns)}")

# Type distribution
if 'type' in pdf.columns:
    print(f"\n  Type distribution:")
    for t in sorted(pdf['type'].unique()):
        print(f"    {t}: {(pdf['type']==t).sum()}")

# =============================================
#  TYPE INFERENCE FUNCTION
# =============================================
import re as _re_type

_NUM_EQ_RE = _re_type.compile(r'^(\d+)([^\d])(\d+)$')

def _is_numeric_equation(prompt):
    """Check if a symbol/equation prompt is numeric (digit-op-digit) vs symbolic."""
    for line in prompt.strip().split('\n'):
        line = line.strip()
        if ' = ' in line and 'alice' not in line.lower() and 'equation' not in line.lower() \
                and 'transformation' not in line.lower() and 'determine' not in line.lower():
            lhs = line.split(' = ', 1)[0].strip()
            if lhs:
                return bool(_NUM_EQ_RE.match(lhs))
    return False

def _infer_type(prompt):
    p = prompt.lower()
    if 'bit manipulation' in p or '8-bit binary' in p:
        return 'bit_ops'
    elif 'numeral system' in p:
        return 'numeral'
    elif 'encrypt' in p or 'decrypt' in p:
        return 'cipher'
    elif 'gravitational' in p or 'gravity' in p or 'free-fall' in p:
        return 'gravity'
    elif 'unit' in p and ('convert' in p or 'conversion' in p):
        return 'unit_conv'
    elif 'symbol' in p or 'transformation rule' in p:
        if _is_numeric_equation(prompt):
            return 'eq_numeric'
        return 'eq_symbolic'
    return 'unknown'

# =============================================
#  HOLDOUT SPLIT (for per-type evaluation)
# =============================================
if HOLDOUT_ENABLED:
    # Load original train.csv for ground truth + type inference
    _orig_train = pl.read_csv(f'{COMP_DATA}/train.csv').to_pandas()
    _orig_train['type'] = _orig_train['prompt'].apply(_infer_type)
    
    if HOLDOUT_HARD_IDS:
        # Use hardcoded holdout IDs (solver-failed but gold-reachable problems)
        holdout_df = _orig_train[_orig_train['id'].isin(HOLDOUT_HARD_IDS)].reset_index(drop=True)
        holdout_ids = set(holdout_df['id'].tolist())
    else:
        # Stratified holdout: HOLDOUT_N_PER_TYPE per TRAIN_TYPE
        _holdout_types = TRAIN_TYPES if TRAIN_TYPES else sorted(_orig_train['type'].unique())
        _holdout_parts = []
        for _t in sorted(_holdout_types):
            _type_df = _orig_train[_orig_train['type'] == _t]
            _sampled = _type_df.sample(n=min(HOLDOUT_N_PER_TYPE, len(_type_df)), random_state=999)
            _holdout_parts.append(_sampled)
        holdout_df = pd.concat(_holdout_parts).reset_index(drop=True)
        holdout_ids = set(holdout_df['id'].tolist())
    
    print(f"\n{'='*60}")
    print(f"  HOLDOUT: {len(holdout_df)} samples ({HOLDOUT_N_PER_TYPE}/type)")
    print(f"{'='*60}")
    for _t in sorted(holdout_df['type'].unique()):
        print(f"    {_t}: {(holdout_df['type']==_t).sum()}")
    
    # Filter training data to exclude holdout
    _before = len(train_df)
    _keep_mask = ~train_df.to_pandas()['id'].isin(holdout_ids)
    train_df = pl.from_pandas(train_df.to_pandas()[_keep_mask.values].reset_index(drop=True))
    print(f"\n  Training data: {_before} → {len(train_df)} (removed {_before - len(train_df)} holdout overlap)")
else:
    holdout_df = None
    print("\nHoldout evaluation: DISABLED")

# =============================================
#  TYPE FILTER (train on subset of types)
# =============================================
if TRAIN_TYPES:
    _train_pdf = train_df.to_pandas()
    _train_pdf['_type'] = _train_pdf['prompt'].apply(_infer_type)
    _before_type = len(_train_pdf)
    _train_pdf = _train_pdf[_train_pdf['_type'].isin(TRAIN_TYPES)].reset_index(drop=True)
    
    print(f"\n{'='*60}")
    print(f"  TYPE FILTER: {TRAIN_TYPES}")
    print(f"  {_before_type} → {len(_train_pdf)} rows")
    print(f"{'='*60}")
    for _t in sorted(_train_pdf['_type'].unique()):
        print(f"    {_t}: {(_train_pdf['_type']==_t).sum()}")
    
    _train_pdf = _train_pdf.drop(columns=['_type'])
    train_df = pl.from_pandas(_train_pdf)
else:
    print("\nType filter: DISABLED (all types)")

## Prompt Format & Training Text

**Single-Stage Format:**
- Prompt suffix + thinking + `\boxed{}` answer all in one stage

The suffix must match the official evaluation metric script EXACTLY:
```python
user_content = item.prompt + '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
```

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# =============================================
#  Prompt suffix verification
# =============================================
print("=== PROMPT SUFFIX VERIFICATION ===")
print(f"Our PROMPT_SUFFIX: {repr(PROMPT_SUFFIX)}")

official_suffix = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
assert PROMPT_SUFFIX == official_suffix, f"❌ MISMATCH!\nOurs:     {repr(PROMPT_SUFFIX)}\nOfficial: {repr(official_suffix)}"
print("✅ Prompt suffix matches official evaluation exactly")

# Show what official eval generates for inference
sample_prompt = "What is 2 + 2?"
official_inference_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": sample_prompt + PROMPT_SUFFIX}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)
print(f"\nOfficial inference prompt (enable_thinking=True):\n---\n{official_inference_prompt}\n---")

# =============================================
#  Helper: check if thinking is empty/missing
# =============================================
def _has_thinking(thinking):
    """Return True if thinking contains actual content (not NaN/None/empty)."""
    if thinking is None:
        return False
    if isinstance(thinking, float) and math.isnan(thinking):
        return False
    s = str(thinking).strip()
    return len(s) > 0 and s.lower() != 'nan'

# =============================================
#  Stage 1 builder: thinking + boxed
# =============================================
def build_stage1_text(example):
    """Stage 1: Build training text. Answer-only or thinking+boxed mode."""
    prompt = example["prompt"]
    answer = str(example["answer"])
    
    if STAGE1_ANSWER_ONLY:
        user_msg = prompt + PROMPT_SUFFIX
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n<think></think>\\boxed{{{answer}}}<|im_end|>"
        )
    else:
        user_msg = prompt + PROMPT_SUFFIX
        thinking = example.get("thinking", None)
        if _has_thinking(thinking):
            text = (
                f"<|im_start|>user\n{user_msg}<|im_end|>\n"
                f"<|im_start|>assistant\n<think>\n{str(thinking).strip()}\n</think>\n\\boxed{{{answer}}}<|im_end|>"
            )
        else:
            text = (
                f"<|im_start|>user\n{user_msg}<|im_end|>\n"
                f"<|im_start|>assistant\n<think></think>\\boxed{{{answer}}}<|im_end|>"
            )
    return {"text": text}

# =============================================
#  Stage 2 builder: thinking + boxed (thinking masked in loss)
# =============================================
THINK_END_TOKEN_ID = 13  # </think> token in Nemotron tokenizer

def build_stage2_text(example):
    """Stage 2: Unified format with thinking + \\boxed{}.
    Thinking content will be masked in loss computation."""
    prompt = example["prompt"]
    answer = str(example["answer"])
    thinking = example.get("thinking", "")
    user_msg = prompt + PROMPT_SUFFIX
    
    if _has_thinking(thinking):
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n<think>\n{str(thinking).strip()}\n</think>\n\\boxed{{{answer}}}<|im_end|>"
        )
    else:
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n<think></think>\\boxed{{{answer}}}<|im_end|>"
        )
    return {"text": text}

def tokenize_and_mask_thinking(example):
    """Tokenize and mask everything up to </think> (inclusive) in labels."""
    text = example['text']
    encoded = tokenizer(text, add_special_tokens=False, truncation=True, max_length=STAGE2_MAX_SEQ)
    input_ids = encoded['input_ids']
    labels = list(input_ids)
    
    try:
        think_end_pos = input_ids.index(THINK_END_TOKEN_ID)
        for i in range(think_end_pos + 1):
            labels[i] = -100
    except ValueError:
        pass
    
    return {
        'input_ids': input_ids,
        'attention_mask': [1] * len(input_ids),
        'labels': labels,
    }

print("\n=== STAGE 1 FORMAT VERIFICATION ===")
sample_df = train_df.to_pandas()

if STAGE1_ANSWER_ONLY:
    row = sample_df.iloc[0].to_dict()
    result = build_stage1_text(row)
    text = result['text']
    print(f"\n--- STAGE 1: ANSWER-ONLY MODE (id={row['id']}) ---")
    print(text[:500])
    assert '<think></think>' in text, "Missing empty think tags"
    assert '\\boxed{' in text, "❌ Answer-only should contain \\boxed{}!"
    assert PROMPT_SUFFIX.lstrip('\n') in text, "❌ Missing prompt suffix!"
    print("✅ Answer-only: has suffix, has \\boxed{}, empty think")
else:
    think_rows = sample_df[sample_df['thinking'].apply(_has_thinking)]
    if len(think_rows) > 0:
        row = think_rows.iloc[0].to_dict()
        result = build_stage1_text(row)
        text = result['text']
        print(f"\n--- STAGE 1: THINKING+BOXED (id={row['id']}) ---")
        print(text[:800])
        if len(text) > 800:
            print(f"... ({len(text)} chars total)")
        assert '<think>\n' in text, "Missing <think> tag"
        assert '\n</think>\n' in text, "Missing </think> tag"
        assert '\\boxed{' in text, "Missing \\boxed{} in thinking+boxed mode!"
        assert PROMPT_SUFFIX.lstrip('\n') in text, "Missing prompt suffix!"
        print("✅ Thinking+boxed: has suffix, has \\boxed{}, has thinking")

    ao_rows = sample_df[~sample_df['thinking'].apply(_has_thinking)]
    if len(ao_rows) > 0:
        row = ao_rows.iloc[0].to_dict()
        result = build_stage1_text(row)
        text = result['text']
        print(f"\n--- STAGE 1: NO-THINKING FALLBACK (id={row['id']}) ---")
        print(text[:500])
        assert '\\boxed{' in text, "Missing \\boxed{} in fallback!"
        assert '<think></think>' in text, "Missing empty think tags"
        print("✅ Fallback: has \\boxed{}, empty think")

print("\n✅ All format checks passed!")

In [ ]:
# Convert dataset for Stage 1
stage1_pdf = train_df.to_pandas()

# Filter by thinking mode
if STAGE1_THINKING_ONLY:
    before = len(stage1_pdf)
    stage1_pdf = stage1_pdf[stage1_pdf['thinking'].apply(_has_thinking)].reset_index(drop=True)
    print(f"Stage 1 thinking-only filter: {before} → {len(stage1_pdf)} rows (dropped {before - len(stage1_pdf)} answer-only)")
elif STAGE1_ANSWER_ONLY:
    print(f"Stage 1 answer-only mode: using all {len(stage1_pdf)} rows (thinking ignored)")

# Stratified per-type sampling with oversampling for under-represented types
if STAGE1_N_PER_TYPE is not None:
    _type_col = stage1_pdf['type'] if 'type' in stage1_pdf.columns else stage1_pdf['prompt'].apply(_infer_type)
    parts = []
    for t in sorted(_type_col.unique()):
        t_df = stage1_pdf[_type_col == t]
        n = STAGE1_N_PER_TYPE
        if len(t_df) >= n:
            parts.append(t_df.sample(n=n, random_state=42))
        else:
            parts.append(t_df.sample(n=n, replace=True, random_state=42))
            print(f"  ⚠️ {t}: oversampled {len(t_df)} → {n}")
    stage1_pdf = pd.concat(parts).reset_index(drop=True)
    print(f"Stratified sampling: {STAGE1_N_PER_TYPE}/type × {len(_type_col.unique())} types = {len(stage1_pdf)} rows")

hf_dataset = Dataset.from_pandas(stage1_pdf)
hf_dataset = hf_dataset.map(
    build_stage1_text,
    remove_columns=hf_dataset.column_names,
)
print(f"Stage 1 dataset formatted: {len(hf_dataset)} rows")

# Token length analysis
print("\n=== TOKEN LENGTH ANALYSIS ===")

token_lengths = []
for i in range(len(hf_dataset)):
    toks = tokenizer(hf_dataset[i]['text'], add_special_tokens=False)
    token_lengths.append(len(toks['input_ids']))

tl = np.array(token_lengths)
print(f"  Total samples: {len(tl)}")
print(f"  Min tokens:    {tl.min()}")
print(f"  Median tokens: {np.median(tl):.0f}")
print(f"  Mean tokens:   {tl.mean():.0f}")
print(f"  P95 tokens:    {np.percentile(tl, 95):.0f}")
print(f"  P99 tokens:    {np.percentile(tl, 99):.0f}")
print(f"  Max tokens:    {tl.max()}")

# Drop samples exceeding max_seq if enabled
if STAGE1_DROP_LONG:
    keep_mask = tl <= STAGE1_MAX_SEQ
    n_drop = (~keep_mask).sum()
    print(f"\n  Dropping {n_drop} samples > {STAGE1_MAX_SEQ} tokens ({n_drop/len(tl)*100:.1f}%)")
    keep_indices = np.where(keep_mask)[0].tolist()
    hf_dataset = hf_dataset.select(keep_indices)
    tl = tl[keep_mask]
    print(f"  Remaining: {len(hf_dataset)} samples (max={tl.max()} tokens, zero truncation)")
else:
    truncated = (tl > STAGE1_MAX_SEQ).sum()
    print(f"\n  Truncated at {STAGE1_MAX_SEQ}: {truncated} ({truncated/len(tl)*100:.1f}%)")

# Show 3 samples: short, medium, long
sorted_idx = np.argsort(tl)
for label, idx in [("SHORTEST", sorted_idx[0]), ("MEDIAN", sorted_idx[len(sorted_idx)//2]), ("LONGEST", sorted_idx[-1])]:
    print(f"\n--- {label} ({tl[idx]} tokens) ---")
    print(hf_dataset[int(idx)]['text'][:400])
    if len(hf_dataset[int(idx)]['text']) > 400:
        print(f"... [truncated, {len(hf_dataset[int(idx)]['text'])} chars]")

## Model Loading with Unsloth

Uses `FastLanguageModel` from Unsloth. This splits Mamba's `in_proj` into `gate_proj` + `x_proj`
with separate LoRA on each — more fine-grained than HF PEFT. Also fuses MoE expert weights for 3D LoRA.

In [ ]:
from unsloth import FastLanguageModel
from unittest.mock import MagicMock

# Mock problematic CUDA modules
_mock_modules = [
    "cutlass", "cutlass.cute", "cutlass.utils",
    "mamba_ssm.ops.cute", "mamba_ssm.ops.cute.mamba3",
    "mamba_ssm.ops.cute.mamba3.mamba3_step_fn",
    "mamba_ssm.ops.tilelang", "mamba_ssm.ops.tilelang.mamba3",
    "mamba_ssm.ops.tilelang.mamba3.mamba3_mimo",
]
for mod_name in _mock_modules:
    if mod_name not in sys.modules:
        sys.modules[mod_name] = MagicMock()

t0 = time.time()

# Load with Unsloth — use LOCAL metric/ model path (no internet needed!)
# Unsloth auto-detects NemotronH architecture and splits Mamba in_proj → gate_proj + x_proj
# Also fuses MoE expert weights (w1/w2) for efficient 3D LoRA
print(f"Loading model with Unsloth from: {MODEL_PATH}")
model, unsloth_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=STAGE1_MAX_SEQ,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    trust_remote_code=True,
    unsloth_force_compile=False,
    attn_implementation="eager",
)
print(f"Model loaded in {time.time()-t0:.1f}s")

# Patch: force slow path for Blackwell
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        if hasattr(mod, 'is_fast_path_available'):
            mod.is_fast_path_available = False
            print(f"Patched {name}: is_fast_path_available = False")

# Setup LoRA with Unsloth
print(f"\nLoRA: r={LORA_RANK}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
# Explicit target_modules to bypass Unsloth's get_peft_regex()
# which fails on metric/ models (backbone.layers vs model.layers regex mismatch).
# NOTE: gate_proj/x_proj (Unsloth-split Mamba) REMOVED — the gate+x → in_proj SVD
# merge in the conversion cell assumes layout [gate | x | ...] for in_proj.weight,
# but Nemotron-H Mamba2 packs in_proj as [z, x, B, C, dt] (gate=z is NOT contiguous
# with x at the start). Wrong rows get the merged LoRA delta → severe degradation
# (~0.57 vs 0.72 expected). in_proj/out_proj are kept and pass through unchanged,
# matching V120's HF PEFT all-linear coverage.
_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",   # Attention
    "up_proj", "down_proj",                     # MoE experts
    "in_proj", "out_proj",                      # Mamba (original, direct LoRA)
    "lm_head",                                  # Output head
]
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=_TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Print trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params/1e9:.2f}B")
print(f"Trainable params: {trainable_params/1e6:.1f}M ({trainable_params/total_params*100:.3f}%)")

# List LoRA modules for verification
print("\n=== LoRA Module Coverage ===")
_lora_modules = set()
for name, param in model.named_parameters():
    if param.requires_grad and 'lora' in name.lower():
        parts = name.split('.')
        for i, p in enumerate(parts):
            if 'lora' in p.lower():
                module_path = '.'.join(parts[max(0,i-2):i])
                _lora_modules.add(module_path)
                break
print(f"Unique LoRA target types: {len(_lora_modules)}")

for m in sorted(_lora_modules)[:20]:

    print(f"  {m}")    print(f"  ... and {len(_lora_modules) - 20} more")
if len(_lora_modules) > 20:

## Training with SFTTrainer + Boxed Loss Weight

In [ ]:
import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

# Import trl AFTER Unsloth to ensure monkey-patching takes effect
from trl import SFTTrainer, SFTConfig

eff_batch = STAGE1_BATCH * STAGE1_GRAD_ACCUM
total_steps = (len(hf_dataset) // eff_batch) * STAGE1_EPOCHS
print(f"{'='*60}")
print(f"  STAGE 1: Reasoning Training (Unsloth)")
print(f"  Samples: {len(hf_dataset)}, LR: {STAGE1_LR}, Epochs: {STAGE1_EPOCHS}")
print(f"  Max Seq: {STAGE1_MAX_SEQ}, Batch: {STAGE1_BATCH}, Grad Accum: {STAGE1_GRAD_ACCUM}")
print(f"  Packing: {STAGE1_PACKING}, Effective batch: {eff_batch}")
print(f"  Estimated steps: ~{total_steps}")
print(f"  Boxed loss weight: {BOXED_LOSS_WEIGHT}")
print(f"{'='*60}")

# NOTE: gradient_checkpointing NOT set here — Unsloth handles it
# via use_gradient_checkpointing="unsloth" in get_peft_model.
# Setting both causes conflicts.
stage1_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=STAGE1_BATCH,
    gradient_accumulation_steps=STAGE1_GRAD_ACCUM,
    num_train_epochs=STAGE1_EPOCHS,
    learning_rate=STAGE1_LR,
    logging_steps=10,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=STAGE1_MAX_SEQ,
    packing=STAGE1_PACKING,
)

# ── Weighted Boxed Loss: upweight final \boxed{answer} tokens ──────────────
_compute_loss_fn = None
if BOXED_LOSS_WEIGHT > 1.0:
    _boxed_marker_ids = tokenizer.encode("\\boxed{", add_special_tokens=False)
    _boxed_weight = float(BOXED_LOSS_WEIGHT)
    print(f"[loss] \\boxed{{ marker tokenizes to {len(_boxed_marker_ids)} token IDs: {_boxed_marker_ids}")

    def _weighted_boxed_loss(outputs, labels, num_items_in_batch=None):
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        batch_size, seq_len = shift_labels.shape

        loss_fct = torch.nn.CrossEntropyLoss(reduction='none', ignore_index=-100)
        per_token_loss = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        ).view(batch_size, seq_len)

        # Build weight mask: upweight tokens after the LAST \boxed{ occurrence
        weights = torch.ones(batch_size, seq_len, device=per_token_loss.device)
        marker = torch.tensor(_boxed_marker_ids, device=shift_labels.device)
        marker_len = len(_boxed_marker_ids)

        for bi in range(batch_size):
            last_pos = -1
            for i in range(seq_len - marker_len + 1):
                if torch.equal(shift_labels[bi, i:i+marker_len], marker):
                    last_pos = i
            if last_pos >= 0:
                weights[bi, last_pos:] = _boxed_weight

        mask = (shift_labels != -100).float()
        weighted_loss = (per_token_loss * weights * mask).sum() / (weights * mask).sum()
        return weighted_loss

    _compute_loss_fn = _weighted_boxed_loss
    print(f"[loss] Weighted boxed loss ready: weight={_boxed_weight}x after last \\boxed{{")
else:
    print("[loss] BOXED_LOSS_WEIGHT <= 1.0 — using default uniform loss")

stage1_trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=stage1_args,
    compute_loss_func=_compute_loss_fn,
)

t0 = time.time()
stage1_result = stage1_trainer.train()
stage1_time = time.time() - t0

print(f"\n{'='*60}")
print(f"  STAGE 1 COMPLETE")
print(f"  Time: {stage1_time/60:.1f} min")
print(f"  Final loss: {stage1_result.training_loss:.4f}")
print(f"{'='*60}")

## Stage 2: Format Polish (Optional)

Light fine-tuning with answer-only data at low LR. Reinforces `\boxed{}` output format without disrupting learned reasoning.

Set `STAGE2_ENABLED = False` above to skip this stage.

In [ ]:
if STAGE2_ENABLED:
    from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
    
    print(f"{'='*60}")
    print(f"  STAGE 2: Format Polish (thinking masked)")
    print(f"{'='*60}")
    
    # Prepare Stage 2 dataset from THINKING rows (not answer-only)
    full_df = train_df.to_pandas()
    think_mask = full_df['thinking'].apply(_has_thinking)
    thinking_df = full_df[think_mask]
    
    n_sample = min(STAGE2_N_SAMPLES, len(thinking_df))
    stage2_df = thinking_df.sample(n=n_sample, random_state=42)
    
    eff_batch2 = STAGE2_BATCH * STAGE2_GRAD_ACCUM
    total_steps2 = (n_sample // eff_batch2) * STAGE2_EPOCHS
    print(f"  Thinking pool: {len(thinking_df)}")
    print(f"  Sampled for Stage 2: {n_sample}")
    print(f"  LR: {STAGE2_LR}, Epochs: {STAGE2_EPOCHS}, Max Seq: {STAGE2_MAX_SEQ}")
    print(f"  Batch: {STAGE2_BATCH}, Grad Accum: {STAGE2_GRAD_ACCUM}")
    print(f"  Loss: thinking MASKED, only \\boxed{{answer}} trained")
    print(f"  Estimated steps: ~{total_steps2}")
    
    # Build text then tokenize with thinking masked
    stage2_raw = Dataset.from_pandas(stage2_df)
    stage2_raw = stage2_raw.map(
        build_stage2_text,
        remove_columns=stage2_raw.column_names,
    )
    stage2_tokenized = stage2_raw.map(
        tokenize_and_mask_thinking,
        remove_columns=['text'],
    )
    
    # Show a Stage 2 sample with masking info
    print(f"\n--- Stage 2 sample ---")
    sample_ids = stage2_tokenized[0]['input_ids']
    sample_labels = stage2_tokenized[0]['labels']
    n_masked = sum(1 for l in sample_labels if l == -100)
    n_trained = sum(1 for l in sample_labels if l != -100)
    print(f"  Tokens: {len(sample_ids)}, masked: {n_masked}, trained: {n_trained}")
    trained_ids = [i for i, l in zip(sample_ids, sample_labels) if l != -100]
    print(f"  Trained text: {tokenizer.decode(trained_ids)[:200]}")
    
    collator = DataCollatorForSeq2Seq(tokenizer, padding=True)
    
    stage2_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=STAGE2_BATCH,
        gradient_accumulation_steps=STAGE2_GRAD_ACCUM,
        num_train_epochs=STAGE2_EPOCHS,
        learning_rate=STAGE2_LR,
        logging_steps=5,
        bf16=True,
        max_grad_norm=1.0,
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        save_strategy="no",
        report_to="none",
        # NOTE: no gradient_checkpointing here, Unsloth handles it
    )
    
    stage2_trainer = Trainer(
        model=model,
        train_dataset=stage2_tokenized,
        data_collator=collator,
        args=stage2_args,
    )
    
    t0 = time.time()
    stage2_result = stage2_trainer.train()
    stage2_time = time.time() - t0
    
    print(f"\n{'='*60}")
    print(f"  STAGE 2 COMPLETE")
    print(f"  Time: {stage2_time/60:.1f} min")
    print(f"  Final loss: {stage2_result.training_loss:.4f}")
    print(f"{'='*60}")
else:
    print("Stage 2 SKIPPED (STAGE2_ENABLED=False)")

## Save & Convert Unsloth → HF PEFT

**IMPORTANT**: Save runs BEFORE holdout evaluation so the adapter is preserved even if eval OOM's.

Conversion steps:
1. **Key rename**: `base_model.model.model` → `base_model.model.backbone`
2. **Expert unfuse**: `experts.w1` → `experts.{i}.up_proj`, `experts.w2` → `experts.{i}.down_proj`
3. **Mamba merge**: `gate_proj` + `x_proj` → `in_proj` via SVD (rank-64 → rank-32, ~98% preserved)
4. **Config rewrite**: explicit target_modules for HF PEFT

In [ ]:
# --- Save raw Unsloth adapter ---
UNSLOTH_DIR = "/kaggle/working/unsloth_adapter"
os.makedirs(UNSLOTH_DIR, exist_ok=True)
model.save_pretrained(UNSLOTH_DIR)
print(f"Unsloth adapter saved to {UNSLOTH_DIR}")
for f in sorted(os.listdir(UNSLOTH_DIR)):
    size = os.path.getsize(os.path.join(UNSLOTH_DIR, f))
    print(f"  {f:40s} {size/1024:.1f} KB")

# --- Load metric/ model key shapes for conversion ---
from safetensors import safe_open
from safetensors.torch import save_file
import glob as _glob

model_keys = {}
for model_sf in _glob.glob(os.path.join(MODEL_PATH, "*.safetensors")):
    with safe_open(model_sf, framework="pt", device="cpu") as f:
        for key in f.keys():
            tensor_slice = f.get_slice(key)
            model_keys[key] = tuple(tensor_slice.get_shape())
print(f"Loaded {len(model_keys)} model key shapes from metric/ model")

# Show base model per-expert module names
_expert_keys = sorted([k for k in model_keys if '.experts.0.' in k])
print(f"\nBase model per-expert sample keys ({len(_expert_keys)} total):")
for k in _expert_keys[:6]:
    print(f"  {k} {model_keys[k]}")

# --- Key rename: Unsloth uses model.layers, metric/ uses backbone.layers ---
def trained_adapter_key_rename(key_name):
    return key_name.replace("base_model.model.model", "base_model.model.backbone")

# --- Convert Unsloth → HF PEFT ---
print("\n=== Converting Unsloth -> HF PEFT ===")

adapter_tensors = {}
with safe_open(os.path.join(UNSLOTH_DIR, "adapter_model.safetensors"), framework="pt", device="cpu") as f:
    for key in f.keys():
        adapter_tensors[key] = f.get_tensor(key)
print(f"Loaded {len(adapter_tensors)} adapter tensors")

# --- Step 1: Separate LoRA keys from non-LoRA keys (modules_to_save) ---
lora_base_names = set()
non_lora_keys = []
for key in adapter_tensors:
    if key.endswith('.lora_A.weight') or key.endswith('.lora_B.weight'):
        base = re.sub(r'\.lora_[AB]\.weight$', '', key)
        lora_base_names.add(base)
    else:
        non_lora_keys.append(key)
        print(f"  Non-LoRA key (skipped): {key} shape={tuple(adapter_tensors[key].shape)}")

print(f"LoRA pairs: {len(lora_base_names)}, Non-LoRA keys: {len(non_lora_keys)}")

# Show some LoRA base name samples to understand naming
_samples = sorted(lora_base_names)
print(f"\nSample LoRA base names:")
for s in _samples[:5]:
    A = adapter_tensors[f"{s}.lora_A.weight"]
    B = adapter_tensors[f"{s}.lora_B.weight"]
    print(f"  {s}  A:{tuple(A.shape)} B:{tuple(B.shape)}")
# Find any 3D tensors (fused experts)
_3d_bases = [b for b in lora_base_names
             if adapter_tensors[f"{b}.lora_A.weight"].dim() == 3
             or adapter_tensors[f"{b}.lora_B.weight"].dim() == 3]
print(f"\n3D (fused expert) LoRA pairs: {len(_3d_bases)}")
if _3d_bases:
    for s in sorted(_3d_bases)[:3]:
        A = adapter_tensors[f"{s}.lora_A.weight"]
        B = adapter_tensors[f"{s}.lora_B.weight"]
        print(f"  {s}  A:{tuple(A.shape)} B:{tuple(B.shape)}")

# --- Step 2: Identify Mamba gate_proj+x_proj pairs for SVD merge ---
mamba_merge_layers = {}
for base in lora_base_names:
    for proj in ("gate_proj", "x_proj"):
        if f".{proj}" in base and ".experts." not in base:
            layer_path = base.rsplit(f".{proj}", 1)[0]
            mamba_merge_layers.setdefault(layer_path, {})[proj] = base
mamba_merge_layers = {k: v for k, v in mamba_merge_layers.items()
                      if 'gate_proj' in v and 'x_proj' in v}
mamba_merge_bases = set()
for projs in mamba_merge_layers.values():
    mamba_merge_bases.update(projs.values())
print(f"Mamba gate_proj+x_proj merge: {len(mamba_merge_layers)} layers")

# --- Step 3: Build expert name mapping (Unsloth fused → base model per-expert) ---
_base_expert_modules = set()
for k in model_keys:
    m = re.search(r'\.experts\.\d+\.(\w+)\.weight$', k)
    if m:
        _base_expert_modules.add(m.group(1))
print(f"Base model per-expert modules: {sorted(_base_expert_modules)}")

_expert_name_map = {}
for mod in _base_expert_modules:
    _expert_name_map[mod] = mod  # direct match
# Unsloth-specific fused names
if 'up_proj' in _base_expert_modules and 'gate_up_proj' not in _base_expert_modules:
    _expert_name_map['gate_up_proj'] = 'up_proj'
if 'up_proj' in _base_expert_modules:
    _expert_name_map['w1'] = 'up_proj'
if 'down_proj' in _base_expert_modules:
    _expert_name_map['w2'] = 'down_proj'
print(f"Expert name map: {_expert_name_map}")

# --- Step 4: Build converted tensors ---
tensors = {}
stats = {'direct': 0, 'unfused_experts': 0, 'mamba_merge': 0}

for base in sorted(lora_base_names):
    if base in mamba_merge_bases:
        continue

    lora_A = adapter_tensors[f"{base}.lora_A.weight"]
    lora_B = adapter_tensors[f"{base}.lora_B.weight"]
    renamed = trained_adapter_key_rename(base)

    # --- 3D tensors = fused expert LoRA → unfuse to per-expert 2D ---
    if lora_A.dim() == 3 or lora_B.dim() == 3:
        if lora_A.dim() == 2:
            lora_A = lora_A.unsqueeze(0).expand(lora_B.shape[0], -1, -1).contiguous()
        elif lora_B.dim() == 2:
            lora_B = lora_B.unsqueeze(0).expand(lora_A.shape[0], -1, -1).contiguous()

        num_experts = lora_A.shape[0]
        fused_module = base.split('.')[-1]
        per_expert_module = _expert_name_map.get(fused_module, fused_module)

        for i in range(num_experts):
            exp_renamed = re.sub(
                rf'\.experts\.{re.escape(fused_module)}$',
                f'.experts.{i}.{per_expert_module}',
                renamed,
            )
            tensors[f"{exp_renamed}.lora_A.weight"] = lora_A[i].contiguous()
            tensors[f"{exp_renamed}.lora_B.weight"] = lora_B[i].contiguous()
        stats['unfused_experts'] += 1
        continue

    # --- Regular 2D: direct rename ---
    tensors[f"{renamed}.lora_A.weight"] = lora_A
    tensors[f"{renamed}.lora_B.weight"] = lora_B
    stats['direct'] += 1

# --- Step 5: Mamba gate_proj + x_proj → in_proj via QR+SVD ---
if mamba_merge_layers:
    print(f"\nMamba gate_proj+x_proj -> in_proj SVD merge ({len(mamba_merge_layers)} layers):")
    for layer_path, projs in sorted(mamba_merge_layers.items()):
        renamed_layer = trained_adapter_key_rename(layer_path)
        in_proj_base = f"{renamed_layer}.in_proj"
        model_in_proj_key = renamed_layer.replace("base_model.model.", "") + ".in_proj.weight"
        in_proj_dim = model_keys[model_in_proj_key][0]

        gate_A = adapter_tensors[f"{projs['gate_proj']}.lora_A.weight"].float()
        gate_B = adapter_tensors[f"{projs['gate_proj']}.lora_B.weight"].float()
        x_A = adapter_tensors[f"{projs['x_proj']}.lora_A.weight"].float()
        x_B = adapter_tensors[f"{projs['x_proj']}.lora_B.weight"].float()
        rank = gate_A.shape[0]

        A_cat = torch.cat([gate_A, x_A], dim=0)
        B_block = torch.zeros(in_proj_dim, 2 * rank)
        B_block[:gate_B.shape[0], :rank] = gate_B
        B_block[gate_B.shape[0]:gate_B.shape[0] + x_B.shape[0], rank:] = x_B

        Q_B, R_B = torch.linalg.qr(B_block)
        Q_A, R_A = torch.linalg.qr(A_cat.T)
        core = R_B @ R_A.T
        U, S, Vh = torch.linalg.svd(core, full_matrices=False)

        k = rank
        new_B = (Q_B @ U[:, :k]) * S[:k].unsqueeze(0)
        new_A = Vh[:k, :] @ Q_A.T
        kept = S[:k].sum().item()
        total = S.sum().item()
        print(f"  {layer_path}: SVD {kept:.2f}/{total:.2f} ({kept/total*100:.1f}%)")

        tensors[f"{in_proj_base}.lora_A.weight"] = new_A
        tensors[f"{in_proj_base}.lora_B.weight"] = new_B
        stats['mamba_merge'] += 1
else:
    print("\nNo Mamba gate_proj+x_proj pairs (Unsloth uses in_proj directly)")

# --- Summary & Verification ---
print(f"\nConversion stats: {stats}")
print(f"Output: {len(tensors)} tensors")

# Verify converted keys match base model structure
_converted_modules = set()
for k in tensors:
    if k.endswith('.lora_A.weight'):
        base_key = re.sub(r'\.lora_A\.weight$', '', k)
        model_equiv = base_key.replace('base_model.model.', '') + '.weight'
        _converted_modules.add(model_equiv)
_matched = sum(1 for m in _converted_modules if m in model_keys)
_unmatched = [m for m in sorted(_converted_modules) if m not in model_keys]
print(f"\nVerification: {_matched}/{len(_converted_modules)} LoRA targets match base model keys")
if _unmatched:
    print(f"⚠️  {len(_unmatched)} UNMATCHED (will be ignored by vLLM):")
    for m in _unmatched[:10]:
        print(f"  {m}")
    if len(_unmatched) > 10:
        print(f"  ... and {len(_unmatched)-10} more")

save_file(tensors, os.path.join(OUTPUT_DIR, "adapter_model.safetensors"))
size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, "adapter_model.safetensors")) / 1024 / 1024
print(f"\nSaved to {OUTPUT_DIR}/adapter_model.safetensors ({size_mb:.1f} MB)")


In [ ]:
# --- Write HF PEFT adapter_config.json ---

# Dynamically determine target_modules from converted tensors
_converted_target_modules = set()
for k in tensors:
    if k.endswith(".lora_A.weight"):
        parts = k.replace(".lora_A.weight", "").split(".")
        _converted_target_modules.add(parts[-1])
_target_modules = sorted(_converted_target_modules)
print(f"Target modules from converted adapter: {_target_modules}")

adapter_config = {
    "alpha_pattern": {},
    "auto_mapping": None,
    "base_model_name_or_path": str(MODEL_PATH),
    "bias": "none",
    "fan_in_fan_out": False,
    "inference_mode": True,
    "init_lora_weights": True,
    "layer_replication": None,
    "layers_pattern": None,
    "layers_to_transform": None,
    "loftq_config": {},
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "megatron_config": None,
    "megatron_core": "megatron.core",
    "modules_to_save": None,
    "peft_type": "LORA",
    "r": LORA_RANK,
    "rank_pattern": {},
    "revision": None,
    "target_modules": _target_modules,
    "task_type": "CAUSAL_LM",
    "use_dora": False,
    "use_rslora": False,
}

config_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
with open(config_path, "w") as f:
    json.dump(adapter_config, f, indent=2)
print(f"Wrote adapter_config.json")
print(f"  r={adapter_config['r']}, alpha={adapter_config['lora_alpha']}")
print(f"  target_modules={adapter_config['target_modules']}")

# --- Package submission.zip ---
zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ["adapter_model.safetensors", "adapter_config.json"]:
        fpath = os.path.join(OUTPUT_DIR, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, fname)
            print(f"  Added {fname} ({os.path.getsize(fpath)/1024/1024:.1f} MB)")

print(f"\nsubmission.zip: {os.path.getsize(zip_path)/1024/1024:.1f} MB")

# Verify
with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    assert "adapter_config.json" in names
    assert "adapter_model.safetensors" in names
    print(f"Contents: {names}")

print("submission.zip ready!")


## Holdout Evaluation (Optional)

In [ ]:
# =============================================
#  HOLDOUT EVALUATION — Per-Type Accuracy
# =============================================
import re as _re

eval_time = 0
results_df = None

# Save training metrics before cleanup
_s1_loss = stage1_result.training_loss if 'stage1_result' in dir() else -1
_s2_loss = stage2_result.training_loss if (STAGE2_ENABLED and 'stage2_result' in dir()) else -1

if HOLDOUT_ENABLED and holdout_df is not None:

    # --- Memory cleanup: free training state before inference ---
    print("Freeing training memory...")
    _g = globals()
    for _obj_name in ['stage1_trainer', 'stage2_trainer', 'stage1_result', 'stage2_result',
                       'hf_dataset', 'stage2_tokenized', 'stage2_raw', 'collator',
                       'stage1_args', 'stage2_args']:
        if _obj_name in _g:
            del _g[_obj_name]
    gc.collect()
    torch.cuda.empty_cache()
    _mem_free = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"GPU free memory after cleanup: {_mem_free:.1f} GB")

    # --- Merge LoRA into base model to save ~8GB VRAM ---
    print("Merging LoRA into base model...")
    model = model.merge_and_unload()
    gc.collect()
    torch.cuda.empty_cache()
    _mem_free2 = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"GPU free memory after merge: {_mem_free2:.1f} GB (saved {_mem_free2 - _mem_free:.1f} GB)")

    print(f"Holdout types: {sorted(holdout_df['type'].unique().tolist())}")

    # --- Official answer extraction (from nemotron-baseline-evaluation.ipynb) ---
    def extract_final_answer(text):
        if text is None:
            return 'NOT_FOUND'
        matches = _re.findall(r'\\boxed\{([^}]*)(?:\}|$)', text)
        if matches:
            non_empty = [m.strip() for m in matches if m.strip()]
            if non_empty:
                return non_empty[-1]
            return matches[-1].strip()
        patterns = [
            r'The final answer is:\s*([^\n]+)',
            r'Final answer is:\s*([^\n]+)',
            r'Final answer\s*[:：]\s*([^\n]+)',
            r'final answer\s*[:：]\s*([^\n]+)',
        ]
        for pattern in patterns:
            matches = _re.findall(pattern, text, _re.IGNORECASE)
            if matches:
                return matches[-1].strip()
        matches = _re.findall(r'-?\d+(?:\.\d+)?', text)
        if matches:
            return matches[-1]
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        return lines[-1] if lines else 'NOT_FOUND'

    # --- Official verify (stricter than competition for honest holdout) ---
    import math as _math
    _BINARY_RE = _re.compile(r'^[01]+$')
    def eval_verify(stored_answer, predicted):
        stored_answer = str(stored_answer).strip()
        predicted = str(predicted).strip()
        # String exact match first
        if predicted.lower() == stored_answer.lower():
            return True
        # Binary strings (e.g. bit_ops "00111111"): require EXACT match only
        # Official eval has a bug where float("00111111") succeeds and tolerance
        # makes near-miss binary strings pass. We enforce strict matching here.
        if len(stored_answer) > 1 and _BINARY_RE.match(stored_answer):
            return False
        # Numeric tolerance for non-binary answers
        try:
            stored_num = float(stored_answer)
            predicted_num = float(predicted)
            return _math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
        except Exception:
            return False

    EVAL_MAX_TOKENS = EVAL_MAX_TOKENS if 'EVAL_MAX_TOKENS' in dir() else 1024

    eval_results = []
    t0_eval = time.time()

    print(f"\n{'='*60}")
    print(f"  HOLDOUT EVAL: {len(holdout_df)} samples")
    print(f"  Types: {sorted(holdout_df['type'].unique().tolist())}")
    print(f"  Decoding: greedy (temperature=0.0) -- matches official eval")
    print(f"  Max new tokens: {EVAL_MAX_TOKENS}")
    print(f"{'='*60}")

    try:
        for eval_idx, row in holdout_df.iterrows():
            prompt = row['prompt']
            answer = str(row['answer'])
            qtype = row['type']

            # Build prompt exactly like official eval
            user_content = prompt + PROMPT_SUFFIX
            chat_prompt = tokenizer.apply_chat_template(
                [{"role": "user", "content": user_content}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True,
            )

            inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)
            prompt_len = inputs['input_ids'].shape[1]

            with torch.no_grad():
                output = model.generate(
                    **inputs,
                    max_new_tokens=EVAL_MAX_TOKENS,
                    do_sample=False,
                )

            gen_ids = output[0][prompt_len:]
            generated = tokenizer.decode(gen_ids, skip_special_tokens=False)
            predicted = extract_final_answer(generated)
            correct = eval_verify(answer, predicted)

            # --- Verbose per-sample output ---
            n_done = len(eval_results) + 1
            status = "✅" if correct else "❌"
            print(f"\n{'='*60}")
            print(f"  [{n_done}/{len(holdout_df)}] {status} {qtype} | expected={answer} | predicted={predicted} | tokens={len(gen_ids)}")
            print(f"{'='*60}")
            print(f"  FULL OUTPUT ({len(generated)} chars):")
            print(generated)
            print(f"{'─'*60}")

            eval_results.append({
                'id': row['id'],
                'type': qtype,
                'answer': answer,
                'predicted': predicted,
                'correct': correct,
                'gen_tokens': len(gen_ids),
                'generated': generated,
            })

            # Free KV cache every sample
            del output, inputs, gen_ids
            torch.cuda.empty_cache()

    except Exception as _eval_err:
        print(f"\n⚠️ Eval stopped after {len(eval_results)}/{len(holdout_df)} samples: {_eval_err}")
        print("Adapter was already saved — no data loss.")

    eval_time = time.time() - t0_eval

    # --- Results table (even if partial) ---
    if eval_results:
        import pandas as pd
        results_df = pd.DataFrame(eval_results)

        print(f"\n{'='*60}")
        _label = "PARTIAL" if len(eval_results) < len(holdout_df) else "FULL"
        print(f"  📊 HOLDOUT RESULTS [{_label}] (eval time: {eval_time/60:.1f} min)")
        print(f"{'='*60}")
        print(f"  {'Type':12s} {'Correct':>8s} {'Total':>6s} {'Acc':>8s} {'Avg Tokens':>11s}")
        print(f"  {'-'*47}")

        for t in sorted(results_df['type'].unique()):
            t_df = results_df[results_df['type'] == t]
            n_correct = t_df['correct'].sum()
            n_total = len(t_df)
            avg_tok = t_df['gen_tokens'].mean()
            acc = n_correct / n_total * 100
            bar = '█' * int(acc / 5) + '░' * (20 - int(acc / 5))
            print(f"  {t:12s} {n_correct:5d}/{n_total:<3d}   {acc:5.1f}%  {avg_tok:8.0f}  {bar}")

        total_correct = results_df['correct'].sum()
        total_n = len(results_df)
        total_acc = total_correct / total_n * 100
        print(f"  {'-'*47}")
        print(f"  {'TOTAL':12s} {total_correct:5d}/{total_n:<3d}   {total_acc:5.1f}%")

        # --- Show failures ---
        failures = results_df[~results_df['correct']]
        if len(failures) > 0:
            print(f"\n  ❌ Failures ({len(failures)} total, showing first 10):")
            for _, f in failures.head(10).iterrows():
                print(f"    [{f['type']:10s}] expected='{f['answer']}', got='{f['predicted']}' ({f['gen_tokens']} tok)")

        # --- Token stats ---
        print(f"\n  Token stats: min={results_df['gen_tokens'].min()}, "
              f"median={results_df['gen_tokens'].median():.0f}, "
              f"max={results_df['gen_tokens'].max()}")
        not_found = (results_df['predicted'] == 'NOT_FOUND').sum()
        if not_found > 0:
            print(f"  ⚠️ NOT_FOUND: {not_found} samples (no \\boxed{{}} in output)")
    else:
        print("No eval results collected.")

else:
    print("Holdout evaluation: SKIPPED (HOLDOUT_ENABLED=False)")

# =============================================
#  TRAINING SUMMARY
# =============================================
print("\n" + "=" * 60)
print("  TRAINING SUMMARY")
print("=" * 60)
print(f"  Stage 1: loss={_s1_loss:.4f}, time={stage1_time/60:.1f}min")
if STAGE2_ENABLED:
    print(f"  Stage 2: {n_sample} samples, loss={_s2_loss:.4f}, time={stage2_time/60:.1f}min")
    print(f"  Total training time: {(stage1_time + stage2_time)/60:.1f} min")
else:
    print(f"  Stage 2: SKIPPED")
    print(f"  Total time: {stage1_time/60:.1f} min")
print(f"  LoRA rank: {LORA_RANK}, alpha: {LORA_ALPHA}")
print(f"  Prompt suffix verified: ✅")
print(f"  Output: /kaggle/working/submission.zip")

if HOLDOUT_ENABLED and results_df is not None and len(results_df) > 0:
    total_correct = results_df['correct'].sum()
    total_n = len(results_df)
    print(f"  Holdout: {total_correct}/{total_n} = {total_correct/total_n*100:.1f}%")
    print(f"  Eval time: {eval_time/60:.1f} min")

print("=" * 60)